In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import seaborn as sns
from src.analyze import analysis
from headcast.headcast_funcs import csv_path_list, get_all_labels, dir_data, print_dict_tree, transform_dict
from headcast.pmtx_funcs import read_pmtx, PMTX
from sklearn.metrics import confusion_matrix
from src.plot import gen_figure
from pprint import pprint
from glob import glob
import contextlib
import os

In [ ]:
label_map = {0:'straight', 1: 'straight', 2:'left turn', 3:'left shallow turn',
                4:'left sharp turn', 5:'right turn', 6:'right shallow turn', 7:'right sharp turn'}
tanish_label_map = {0:'straight', 1: '', 2:'left turn',
                3:'left turn', 4:'', 5:'right turn', 6:'right turn'}

HL = '/Users/hind/Documents/UCSB/Neuroscience/headcasting_project/working_copy/headlocked'
output_dir = '/Users/hind/Documents/UCSB/Neuroscience/headcasting_project/working_copy/headlocked/figures/pmtx'
cond = 'headlocked'

In [ ]:
def get_analysis_data(*csv_path_lists, gt_string=None, col=2, 
                    invert=False, outward=False):
    all_info = {}
    if len(csv_path_lists) < 2:
        raise ValueError("At least two lists are required")
    min_len = min(len(lst) for lst in csv_path_lists)
    for i in range(min_len):
        # Unpack the i-th element from each list
        csvs = [lst[i] for lst in csv_path_lists]
        # print(csvs)
        # Pass all csvs to analysis using *
        info = analysis(*csvs, col=col, gt_string=gt_string, show_plot=False)
        print(info.keys())
        for key in info:
            if key not in all_info and '.csv' in key:
                all_info[key] = info[key]
            if key == 'comparison_results':
                all_info[i+1] = info[key]

    print("All info keys:")
    pprint(all_info.keys())
    transformed_info = transform_dict(all_info, invert=invert, outward_only=outward)
    return transformed_info

# Define paths to label dirs

In [ ]:
nitesh_hl = os.path.join(HL, 'Nitesh')
tanish_hl50 = os.path.join(HL, 'Tanish', '50thQ')
tanish_hl75 = os.path.join(HL, 'Tanish', '75thQ')

peris = csv_path_list(nitesh_hl, outward_only=True, 
                    invert=True, raw=True) #only use outward casts, invert left/right

q50 = csv_path_list(tanish_hl50, raw=True)
q75 = csv_path_list(tanish_hl75, raw=True)

In [ ]:
csv_list1 = q50
label1 = 'supervised ML - 50q'

csv_list2 = peris
label2 = 'peristalsis'

csv_list3 = q75
label3 = 'supervised ML - 75q'

In [ ]:
annot_labels = ['straight', 'cast', 'turn', 'cast + turn']
final_cm = np.zeros((len(annot_labels), len(annot_labels)), dtype=int)

for i in range(len(csv_list1)):
    pmtx1 = PMTX(csv_list1[i])
    pmtx2 = PMTX(csv_list2[i], invert=True)
    pmtx3 = PMTX(csv_list3[i])

    final_cm += confusion_matrix(pmtx2.directionless_labels, pmtx1.directionless_labels, labels=[0,2,3,4])

final_cm = np.round(final_cm,1)

fig, ax = plt.subplots()
sns.heatmap(final_cm, annot=True, fmt='.2f', xticklabels=annot_labels, 
            yticklabels=annot_labels, cmap='Blues', ax=ax)
ax.set_xlabel('Tanish')
ax.xaxis.set_label_position('top')
ax.xaxis.tick_top()
plt.ylabel('Nitesh')
plt.show()

In [ ]:
all_outward = []
all_centerline = []
for i in range(len(csv_list3)):
    df = read_pmtx(csv_list3[i])
    # print(f"Processing file: {os.path.basename(csv_list2[i])}")
    all_outward.append(df['outward'].unique())
    all_centerline.append(df['centerline'].unique())

print("Outward unique values:",np.unique(np.concatenate(all_outward)))
print("Centerline unique values:", np.unique(np.concatenate(all_centerline)))

In [ ]:
lists = [csv_list1, csv_list2, csv_list3]
labels = [label1, label2, label3]

for i, l in enumerate(lists):
    n_casts = 0
    n_cycles = 0
    n_outward = 0
    n_centerline = 0
    outcast = 0
    for j in range(len(l)):
        df = read_pmtx(l[j])
        # print(f"Processing file: {os.path.basename(l[j])}")
        n_casts += df['is_cast'].sum()
        n_cycles += len(df['is_cast'])
        n_outward += df['outward'].sum()
        n_centerline += df['centerline'].sum()
        outc = np.logical_and(df['is_cast'], df['outward'])
        outcast += np.logical_and(outc, df['centerline']).sum()

    print(f"{labels[i]}: {n_casts} casts, {n_cycles} cycles, {n_outward} outward, {n_centerline} centerline")
    print(f"{labels[i]}: {outcast} outward casts")

## rules

In [ ]:
# outward_cast = np.logical_and(df['is_cast'], df['outward'])
# cast = np.logical_and(outward_cast, df['centerline'])

# turn = df['is_turn']
# cast_turn = np.logical_and(turn, cast)

# left_turn = np.logical_and(turn, df['turn_direction'] == 1)
# right_turn = np.logical_and(turn, df['turn_direction'] == -1)

# left_cast = np.logical_and(cast, df['cast_direction'] == 1)
# right_cast = np.logical_and(cast, df['cast_direction'] == -1)

# turn_only = np.logical_and(turn, ~cast)
# turn_only_left = np.logical_and(turn_only, df['turn_direction'] == 1)
# turn_only_right = np.logical_and(turn_only, df['turn_direction'] == -1)

# cast_only = np.logical_and(cast, ~turn)
# cast_only_left = np.logical_and(cast_only, df['cast_direction'] == 1)
# cast_only_right = np.logical_and(cast_only, df['cast_direction'] == -1)

# left_cast_turn = np.logical_and(left_cast, left_turn)
# right_cast_turn = np.logical_and(right_cast, right_turn)

# left_cast_right_turn = np.logical_and(left_cast, right_turn)
# right_cast_left_turn = np.logical_and(right_cast, left_turn)

# straight = np.logical_and(~turn, ~cast)

In [ ]:
print(outward.values)

In [ ]:
thl = get_all_labels(q50, col=2)
print(np.unique(thl))

thl = get_all_labels(q50, col=[3,4])
print(np.unique(thl))

In [ ]:
c6 = get_all_labels(q50, col=6)
print(np.unique(c6))

## generate histograms

In [ ]:
print(len(csv_list1), len(csv_list2))

In [ ]:
all1 = get_all_labels(csv_list1, col=6)
all2 = get_all_labels(csv_list2, col=6)
print("All labels in csv_list1:", np.unique(all1))
print("All labels in csv_list2:", np.unique(all2))
print(len(all1), len(all2))

print(len(all1[all1 == 0]), len(all2[all2 == 0]))

In [ ]:
# cast/turn info with no direction
bins=[-0.45, 0.45, 0.55, 1.45, 1.55, 2.45, 2.55, 3.45, 3.55, 4.45]
plt.figure(figsize=(8, 5))
plt.hist(get_all_labels(csv_list1, col=6), bins=bins, alpha=0.5, label=label1)
plt.hist(get_all_labels(csv_list2, col=6), bins=bins, alpha=0.5, label=label2)
# plt.hist(get_all_labels(csv_list2, col=6), bins=bins, alpha=0.5, label=label2)
plt.xticks([0, 2, 3, 4],  ['straight', 'cast', 'turn', 'cast + turn'])
plt.title('Merged directionless cast and turn labels')
plt.legend()
plt.show()

In [ ]:
print(np.unique(get_all_labels(csv_list1, col=6)))
print(np.unique(get_all_labels(csv_list2, col=6)))

In [ ]:
all1 = get_all_labels(csv_list1, col=[3,4])
all2 = get_all_labels(csv_list2, col=[3,4])
print("All labels in csv_list1:", np.unique(all1))
print("All labels in csv_list2:", np.unique(all2))
print(len(all1), len(all2))

print(len(all1[all1 == 0]), len(all2[all2 == 0]))

In [ ]:
# merged cast/turn info
bins=[-0.45, 0.45, 0.55, 1.45, 1.55, 2.45, 2.55, 3.45, 3.55, 4.45, 4.55, 5.45, 
    5.55, 6.45, 6.55, 7.45, 7.55, 8.45, 8.55, 9.45, 9.55, 10.45, 10.55, 11.45]
plt.figure(figsize=(15, 5))
plt.hist(get_all_labels(csv_list1, col=[3,4]), bins=bins, alpha=0.5, label=label1)
plt.hist(get_all_labels(csv_list2, col=[3,4]), bins=bins, alpha=0.5, label=label2)
plt.xticks([0, 2, 3, 5, 6, 7, 8, 9, 11],  ['straight', 'left cast', 'left turn', 'right cast', 
                  'right turn', 'left cast\nright turn', 'right cast\nleft turn', 'left cast\nleft turn', 'right cast\nright turn'])
plt.title('Merged cast and turn labels')
plt.legend()
plt.show()

In [ ]:
# accept/reject figure
bin_trio = [-1.45, -0.55, -0.45, 0.45, 0.55, 1.45]

plt.hist(get_all_labels(csv_list1, col=5), bins=bin_trio, alpha=0.5, label=label1)
plt.hist(get_all_labels(csv_list2, col=5), bins=bin_trio, alpha=0.5, label=label2)
plt.xticks([-1, 0, 1], ['reject', 'N/A', 'accept'])
plt.legend()
plt.show()

In [ ]:
# labels for cast and turn separately (effectively reading in 2 behavior columns)
bins = [-0.45, 0.45, 0.55, 1.45, 1.55, 2.45, 2.55, 3.45, 3.55, 4.45, 4.55, 5.45, 5.55, 6.45]
list1 = np.concatenate([get_all_labels(csv_list1, col=3) , get_all_labels(csv_list1, col=4)])
list2 = np.concatenate([get_all_labels(csv_list2, col=3), get_all_labels(csv_list2, col=4)])
plt.hist(list1, bins=bins, alpha=0.5, label=label1)
plt.hist(list2, bins=bins, alpha=0.5, label=label2)
plt.xticks([ 0, 2, 3, 5, 6], [ 'straight', 'left\ncast', 'left\nturn', 'right\ncast', 'right\nturn'])
plt.ylim(0, 9000)
plt.title('Cast and turn labels separately')
plt.legend()
plt.show()

In [ ]:
print(len(csv_list1), len(csv_list2), len(csv_list3))

In [ ]:
casts1 = get_all_labels(csv_list1, col=3)
casts2 = get_all_labels(csv_list2, col=3)

print(np.logical_or(casts1 == 2, casts1 == 5).sum())
print(np.logical_or(casts2 == 2, casts2 == 5).sum())

turns1 = get_all_labels(csv_list1, col=4)
turns2 = get_all_labels(csv_list2, col=4)

print(np.logical_or(turns1 == 3, turns1 == 6).sum())
print(np.logical_or(turns2 == 3, turns2 == 6).sum())

In [ ]:
#cast label histogram
bins = [-0.45, 0.45, 0.55, 1.45, 1.55, 2.45, 2.55, 3.45, 3.55, 4.45, 4.55, 5.45]

plt.hist(get_all_labels(csv_list1, col=3), bins=bins, alpha=0.5, label=label1)
plt.hist(get_all_labels(csv_list2, col=3), bins=bins, alpha=0.5, label=label2)
plt.xticks([0, 2, 5], ['no cast', 'Left cast', 'Right cast'])
plt.legend()
plt.show()

In [ ]:
#turn label histogram
bins = [-0.45, 0.45, 0.55, 1.45, 1.55, 2.45, 2.55, 3.45, 3.55, 4.45, 4.55, 5.45, 5.55, 6.45]

plt.hist(get_all_labels(csv_list1, col=4), bins=bins, alpha=0.5, label=label1)
plt.hist(get_all_labels(csv_list2, col=4), bins=bins, alpha=0.5, label=label2)
plt.xticks([0, 3, 6], ['no turn', 'Left turn', 'Right turn'])
plt.legend()
plt.show()

In [ ]:
# mutually exclusive behavior labels
bins = [-0.45, 0.45, 0.55, 1.45, 1.55, 2.45, 2.55, 3.45, 3.55, 4.45, 4.55, 5.45, 5.55, 6.45]

plt.hist(get_all_labels(csv_list1), bins=bins, alpha=0.5, label= label1)
plt.hist(get_all_labels(csv_list2), bins=bins, alpha=0.5, label= label2)
plt.xticks([ 0, 2, 3, 5, 6], [ 'straight', 'left\ncast', 'left\nturn', 'right\ncast', 'right\nturn'])
plt.legend()
# plt.ylim(0, 9000)
plt.title(f'Mutually exclusive casts and turns')
plt.show()


## generate box plots

In [ ]:
def get_merged_labels(csv_list, col=[3,4]):
    all_annotations = []
    if isinstance(col, (list, tuple)) and len(col) == 2:
        print(f'Merging two columns, {col}')
        for file in csv_list:
            # print(os.path.basename(file))
            df = pd.read_csv(file, header=None)
            col1 = df[col[0]].values
            col2 = df[col[1]].values
            merged =  np.zeros(len(col1))

            # special_case = ((col1 == 2) & (col2 == 6)) | ((col1 == 6) & (col2 == 2))
            special_case2 = ((col1 == 3) & (col2 == 2)) | ((col1 == 2) & (col2 == 3))
            # merged[special_case] = 7  # Special case: set merged to 7
            # merged[special_case2] = 9  # Special case2: set merged to 9
            merged[special_case2] = 11
            non_special = ~special_case2
            merged[non_special] = col1[non_special] + col2[non_special]
            
            all_annotations.append(merged)
    return all_annotations

In [ ]:
def get_traj_labels(file, col=[3,4]):
    if isinstance(col, int):
        # print(os.path.basename(file))
        df = pd.read_csv(file, header=None)
        col1 = df[col].values
        return col1

    if isinstance(col, (list, tuple)) and len(col) == 2:
        # print(f'Merging two columns, {col}')
        # print(os.path.basename(file))
        df = pd.read_csv(file, header=None)
        col1 = df[col[0]].values
        col2 = df[col[1]].values
        merged =  np.zeros(len(col1))

        special_case = (
            (col1 == 2) & (col2 == 6)
            )| (
            (col1 == 6) & (col2 == 2))
        special_case2 = (
            (col1 == 3) & (col2 == 2)
            ) | (
            (col1 == 2) & (col2 == 3))
        
        merged[special_case] = 7  # Special case: set merged to 7
        merged[special_case2] = 9  # Special case2: set merged to 9
        
        # merged[special_case2] = 11
        non_special = ~special_case2 & ~special_case
        merged[non_special] = col1[non_special] + col2[non_special]
        return merged

In [ ]:
print([int(len(get_traj_labels(f, col=[3,4]))) for f in csv_list1])
print([int(len(get_traj_labels(f, col=[3,4]))) for f in csv_list2])

print([int(len(get_traj_labels(f, col=6))) for f in csv_list1])
print([int(len(get_traj_labels(f, col=6))) for f in csv_list2])


In [ ]:
csv_labels1_c2 = [get_traj_labels(file, col=2) for file in csv_list1]
csv_labels1_c34 = [get_traj_labels(file, col=[3,4]) for file in csv_list1]
csv_labels1_c3 = [get_traj_labels(file, col=3) for file in csv_list1]
csv_labels1_c4 = [get_traj_labels(file, col=4) for file in csv_list1]
csv_labels1_c5 = [get_traj_labels(file, col=5) for file in csv_list1]
csv_labels1_c6 = [get_traj_labels(file, col=6) for file in csv_list1]

csv_labels2_c2 = [get_traj_labels(file, col=2) for file in csv_list2]
csv_labels2_c34 = [get_traj_labels(file, col=[3,4]) for file in csv_list2]
csv_labels2_c3 = [get_traj_labels(file, col=3) for file in csv_list2]
csv_labels2_c4 = [get_traj_labels(file, col=4) for file in csv_list2]
csv_labels2_c5 = [get_traj_labels(file, col=5) for file in csv_list2]
csv_labels2_c6 = [get_traj_labels(file, col=6) for file in csv_list2]

csv_labels3_c2 = [get_traj_labels(file, col=2) for file in csv_list3]
csv_labels3_c34 = [get_traj_labels(file, col=[3,4]) for file in csv_list3]
csv_labels3_c3 = [get_traj_labels(file, col=3) for file in csv_list3]
csv_labels3_c4 = [get_traj_labels(file, col=4) for file in csv_list3]
csv_labels3_c5 = [get_traj_labels(file, col=5) for file in csv_list3]
csv_labels3_c6 = [get_traj_labels(file, col=6) for file in csv_list3]

In [ ]:
# mutually exclusive behavior cycles
counts_3 = {0: [],  2: [], 3: [],  5: [], 6: []}
counts_2 = {0: [],  2: [], 3: [],  5: [], 6: []}
counts_1 = {0: [],  2: [], 3: [],  5: [], 6: []}

for key in counts_3.keys():
    for traj in csv_labels3_c2:
        counts_3[key].append(np.sum(traj == key))
    for traj in csv_labels2_c2:
        counts_2[key].append(np.sum(traj == key))
    for traj in csv_labels1_c2:
        counts_1[key].append(np.sum(traj == key))

plt.figure(figsize=(8, 6))
box1 = plt.boxplot([counts_3[key] for key in counts_3.keys()], 
                   positions=[0.2, 2.2, 3.2, 5.2, 6.2], widths=0.15, patch_artist=True, 
                   showfliers=False, label=label3)
for patch in box1['boxes']:
    patch.set_facecolor('#ADD8E6')  # Set color for the first boxplot

box4 = plt.boxplot([counts_1[key] for key in counts_1.keys()],
                   positions=[0.4, 2.4, 3.4, 5.4, 6.4], widths=0.15,
                   patch_artist=True, showfliers=False, label = label1)
for patch in box4['boxes']:
    patch.set_facecolor('#FFB6C1')  # Set color for the third boxplot

# Create the third boxplot and set its color
box3 = plt.boxplot([counts_2[key] for key in counts_2.keys()], 
                   positions=[0.6, 2.6, 3.6, 5.6, 6.6], widths=0.15, 
                   patch_artist=True, showfliers=False, label=label2)
for patch in box3['boxes']:
    patch.set_facecolor('#FFD700')


plt.xticks([0.3, 2.3, 3.3, 5.3, 6.3], 
           labels = ['straight', 'left cast', 'left turn', 'right cast', 'right turn'])
plt.legend()
plt.title(f'Number of mutually exclusive behavioral cycles - {cond}')
plt.ylabel('Number of cycles per trajectory')
plt.show()

In [ ]:
counts_3 = {0: [],  2: [],  5: []}
counts_2 = {0: [],  2: [],  5: []}
counts_1 = {0: [],  2: [],  5: []}

for key in counts_3.keys():
    for traj in csv_labels3_c3:
        counts_3[key].append(np.sum(traj == key))
    for traj in csv_labels2_c3:
        counts_2[key].append(np.sum(traj == key))
    for traj in csv_labels1_c3:
        counts_1[key].append(np.sum(traj == key))

plt.figure(figsize=(8, 6))
box1 = plt.boxplot([counts_3[key] for key in counts_3.keys()], 
                   positions=[0.2, 2.2, 5.2], widths=0.15, patch_artist=True, 
                   showfliers=False, label=label3)
for patch in box1['boxes']:
    patch.set_facecolor('#ADD8E6')  # Set color for the first boxplot

box4 = plt.boxplot([counts_1[key] for key in counts_1.keys()],
                   positions=[0.4, 2.4, 5.4], widths=0.15,
                   patch_artist=True, showfliers=False, label = label1)
for patch in box4['boxes']:
    patch.set_facecolor('#FFB6C1')  # Set color for the third boxplot

# Create the third boxplot and set its color
box3 = plt.boxplot([counts_2[key] for key in counts_2.keys()], 
                   positions=[0.6, 2.6, 5.6], widths=0.15, 
                   patch_artist=True, showfliers=False, label=label2)
for patch in box3['boxes']:
    patch.set_facecolor('#FFD700')


plt.xticks([0.3, 2.3, 5.3], 
           labels = ['straight', 'left cast', 'right cast',])
plt.legend()
plt.title(f'Number of cast cycles - {cond}')
plt.ylabel('Number of cycles per trajectory')
plt.show()

In [ ]:
counts_3 = {0: [],  3: [],  6: []}
counts_2 = {0: [],  3: [],  6: []}
counts_1 = {0: [],  3: [],  6: []}

for key in counts_3.keys():
    for traj in csv_labels3_c4:
        counts_3[key].append(np.sum(traj == key))
    for traj in csv_labels2_c4:
        counts_2[key].append(np.sum(traj == key))
    for traj in csv_labels1_c4:
        counts_1[key].append(np.sum(traj == key))

plt.figure(figsize=(8, 6))
box1 = plt.boxplot([counts_3[key] for key in counts_3.keys()], 
                   positions=[0.2,3.2, 6.2], widths=0.15, patch_artist=True, 
                   showfliers=False, label=label3)
for patch in box1['boxes']:
    patch.set_facecolor('#ADD8E6')  # Set color for the first boxplot

box4 = plt.boxplot([counts_1[key] for key in counts_1.keys()],
                   positions=[0.4, 3.4, 6.4], widths=0.15,
                   patch_artist=True, showfliers=False, label = label1)
for patch in box4['boxes']:
    patch.set_facecolor('#FFB6C1')  # Set color for the third boxplot

# Create the third boxplot and set its color
box3 = plt.boxplot([counts_2[key] for key in counts_2.keys()], 
                   positions=[0.6, 3.6, 6.6], widths=0.15, 
                   patch_artist=True, showfliers=False, label=label2)
for patch in box3['boxes']:
    patch.set_facecolor('#FFD700')


plt.xticks([0.3,  3.3, 6.3], 
           labels = ['straight', 'left turn',  'right turn'])
plt.legend()
plt.title(f'Number of turn cycles - {cond}')
plt.ylabel('Number of cycles per trajectory')
plt.show()

In [ ]:
# non mutually exclusive behavior cycles
counts_3 = {0: [],  2: [], 3: [],  5: [], 6: [], 7: [], 8: [], 9: [],  11: []}
counts_2 = {0: [],  2: [], 3: [],  5: [], 6: [], 7: [], 8: [], 9: [],  11: []}
counts_1 = {0: [],  2: [], 3: [],  5: [], 6: [], 7: [], 8: [], 9: [],  11: []}

for key in counts_3.keys():
    for traj in csv_labels3_c34:
        counts_3[key].append(np.sum(traj == key))
    for traj in csv_labels2_c34:
        counts_2[key].append(np.sum(traj == key))
    for traj in csv_labels1_c34:
        counts_1[key].append(np.sum(traj == key))

plt.figure(figsize=(14, 6))
box1 = plt.boxplot([counts_3[key] for key in counts_3.keys()], 
                   positions=[0.2, 2.2, 3.2, 5.2, 6.2, 7.2, 8.2, 9.2, 11.2], widths=0.15, patch_artist=True, 
                   showfliers=False, label=label3)
for patch in box1['boxes']:
    patch.set_facecolor('#ADD8E6')  # Set color for the first boxplot

box4 = plt.boxplot([counts_1[key] for key in counts_1.keys()],
                   positions=[0.4, 2.4, 3.4, 5.4, 6.4, 7.4, 8.4, 9.4, 11.4], widths=0.15,
                   patch_artist=True, showfliers=False, label = label1)
for patch in box4['boxes']:
    patch.set_facecolor('#FFB6C1')  # Set color for the third boxplot

# Create the third boxplot and set its color
box3 = plt.boxplot([counts_2[key] for key in counts_2.keys()], 
                   positions=[0.6, 2.6, 3.6, 5.6, 6.6, 7.6, 8.6, 9.6, 11.6], widths=0.15, 
                   patch_artist=True, showfliers=False, label=label2)
for patch in box3['boxes']:
    patch.set_facecolor('#FFD700')


plt.xticks([0.3, 2.3, 3.3, 5.3, 6.3, 7.3, 8.3, 9.3, 11.3], 
           labels = ['straight', 'left cast', 'left turn', 'right cast', 'right turn', 
            'left cast\nright turn', 'right cast\nleft turn', 'left cast\nleft turn', 'right cast\nright turn'])
plt.hlines(0, -0.5, 12, colors='black', linestyles='dashed', alpha=0.1)
plt.xlim(-0.5, 12)
plt.legend()
plt.title(f'Number of (merged) behavioral cycles - {cond}')
plt.ylabel('Number of cycles per trajectory')
plt.show()

In [ ]:
counts_3 = {0: [],  1: [],  -1: []}
counts_2 = {0: [],  1: [],  -1: []}
counts_1 = {0: [],  1: [],  -1: []}

for key in counts_3.keys():
    for traj in csv_labels3_c5:
        counts_3[key].append(np.sum(traj == key))
    for traj in csv_labels2_c5:
        counts_2[key].append(np.sum(traj == key))
    for traj in csv_labels1_c5:
        counts_1[key].append(np.sum(traj == key))

plt.figure(figsize=(8, 6))
box1 = plt.boxplot([counts_3[key] for key in counts_3.keys()], 
                   positions=[-0.2 ,1.2, -1.6], widths=0.15, patch_artist=True, 
                   showfliers=False, label=label3)
for patch in box1['boxes']:
    patch.set_facecolor('#ADD8E6')  # Set color for the first boxplot

box4 = plt.boxplot([counts_1[key] for key in counts_1.keys()],
                   positions=[0., 1.4, -1.4], widths=0.15,
                   patch_artist=True, showfliers=False, label = label1)
for patch in box4['boxes']:
    patch.set_facecolor('#FFB6C1')  # Set color for the third boxplot

# Create the third boxplot and set its color
box3 = plt.boxplot([counts_2[key] for key in counts_2.keys()], 
                   positions=[0.2, 1.6, -1.2], widths=0.15, 
                   patch_artist=True, showfliers=False, label=label2)
for patch in box3['boxes']:
    patch.set_facecolor('#FFD700')

plt.xticks([0.,  1.4, -1.4], 
           labels = ['N/A', 'accept',  'reject'])
plt.legend()
plt.title(f'Number of accepted cycles - {cond}')
plt.ylabel('Number of cycles per trajectory')
plt.show()

In [ ]:
counts_3 = {0: [], 2: [], 3:[], 4: []}
counts_2 = {0: [], 2: [], 3:[], 4: []}
counts_1 = {0: [], 2: [], 3:[], 4: []}

for key in counts_3.keys():
    for traj in csv_labels3_c6:
        counts_3[key].append(np.sum(traj == key))
    for traj in csv_labels2_c6:
        counts_2[key].append(np.sum(traj == key))
    for traj in csv_labels1_c6:
        counts_1[key].append(np.sum(traj == key))

plt.figure(figsize=(8, 6))
box1 = plt.boxplot([counts_3[key] for key in counts_3.keys()], 
                   positions=[0.2, 2.2, 3.2, 4.2], widths=0.15, patch_artist=True, 
                   showfliers=False, label=label3)
for patch in box1['boxes']:
    patch.set_facecolor('#ADD8E6')  # Set color for the first boxplot

box4 = plt.boxplot([counts_1[key] for key in counts_1.keys()],
                   positions=[0.4, 2.4, 3.4, 4.4], widths=0.15,
                   patch_artist=True, showfliers=False, label = label1)
for patch in box4['boxes']:
    patch.set_facecolor('#FFB6C1')  # Set color for the third boxplot

# Create the third boxplot and set its color
box3 = plt.boxplot([counts_2[key] for key in counts_2.keys()], 
                   positions=[0.6, 2.6, 3.6, 4.6], widths=0.15, 
                   patch_artist=True, showfliers=False, label=label2)
for patch in box3['boxes']:
    patch.set_facecolor('#FFD700')

plt.xticks([0.3, 2.3, 3.3, 4.3], 
           labels = ['straight', 'cast',  'turn', 'cast + turn'])
plt.legend()
plt.title(f'Number of behavior cycles - {cond}')
plt.ylabel('Number of cycles per trajectory')
plt.show()

In [ ]:
labels1 = get_merged_labels(csv_list1)
labels2 = get_merged_labels(csv_list2)
labels3 = get_merged_labels(csv_list3)

In [ ]:
for i in range(len(labels1)):
    behavior8_list1 = np.sum(labels1[i] == 8)
    q50_num =  behavior8_list1
    behavior8_list2 = np.sum(labels2[i] == 8)
    peris_num =  behavior8_list2
    behavior8_list3 = np.sum(labels3[i] == 8)
    q75_num = behavior8_list3

    max_diff  = max(peris_num - q50_num, peris_num - q75_num)
    if max_diff > 22:
        print(f'File {i+1}: Q50: {q50_num}, Peris: {peris_num}, Q75: {q75_num}, max diff: {max_diff}')

In [ ]:
for i in range(len(labels1)):
    behavior8peris = np.where(labels2[i] == 8)[0]
    cycles =  len(behavior8peris)

    q50_behavior8 = labels1[i][behavior8peris]

    q50_behaviors =  np.sum(q50_behavior8 == 8)
    print(f'File {i+1}: Peris cycles of behavior 7/8: {cycles}, Q50 cycles: {q50_behaviors} - diff: {cycles - q50_behaviors}')

    q75_behavior8 = labels3[i][behavior8peris]
    q75_behaviors = np.sum(q75_behavior8 == 8)
    print(f'File {i+1}: Q75 cycles of behavior 7/8: {q75_behaviors} - diff: {cycles - q75_behaviors}')


## generate ethograms

In [ ]:
# for i in range(len(csv_list1)):
#     # print(np.unique(get_all_labels(c_q50[i], col=2)), np.unique(get_all_labels(c_q625[i], col=2)), np.unique(get_all_labels(c_q75[i], col=2)), np.unique(get_all_labels(peri[i], col=2)))
#     fig = gen_figure(csv_list1[i], csv_list3[i], csv_list2[i], use_cols=3, return_fig=True, titles=[label1, label3, label2])
#     plt.savefig(os.path.join(output_dir, f'{cond}_{i+1}_casts.png'), dpi = 150)
#     plt.close()

#     fig = gen_figure(csv_list1[i], csv_list3[i], csv_list2[i], use_cols=4, return_fig=True, titles=[label1, label3, label2])
#     plt.savefig(os.path.join(output_dir, f'{cond}_{i+1}_turns.png'), dpi = 150)
#     plt.close()

#     fig = gen_figure(csv_list1[i], csv_list3[i], csv_list2[i], use_cols=2, return_fig=True, titles=[label1, label3, label2])
#     plt.savefig(os.path.join(output_dir, f'{cond}_{i+1}_col2.png'), dpi = 150)
#     plt.close()

#     fig = gen_figure(csv_list1[i], csv_list3[i], csv_list2[i], use_cols=[3,4], return_fig=True, titles=[label1, label3, label2])
#     plt.savefig(os.path.join(output_dir, f'{cond}_{i+1}_merged.png'), dpi = 150)
#     plt.close()

#     fig = gen_figure(csv_list1[i], csv_list3[i], csv_list2[i], use_cols=5, return_fig=True, titles=[label1, label3, label2])
#     plt.savefig(os.path.join(output_dir, f'{cond}_{i+1}_accept.png'), dpi = 150)
#     plt.close()
  
#     fig = gen_figure(csv_list1[i], csv_list3[i], csv_list2[i], use_cols=6, return_fig=True, titles=[label1, label3, label2])
#     plt.savefig(os.path.join(output_dir, f'{cond}_{i+1}_directionless.png'), dpi = 150)
#     plt.close()

# compute metrics

In [ ]:
with open(os.devnull, 'w') as f, contextlib.redirect_stdout(f):
    gtstr = 'invert'
    all_info12 = get_analysis_data(csv_list1, csv_list2, gt_string=gtstr,
                                invert=True, outward=True)
    cast_info12 = get_analysis_data(csv_list1, csv_list2, col=3, 
                                gt_string=gtstr, invert=True, outward=True)
    turn_info12 = get_analysis_data(csv_list1, csv_list2, col=4, gt_string=gtstr,
                                invert=True, outward=True)
    accept_info12 = get_analysis_data(csv_list1, csv_list2, col=5, gt_string=gtstr,
                                invert=True, outward=True)
    dirless_info12 = get_analysis_data(csv_list1, csv_list2, col=6, gt_string=gtstr,
                                invert=True, outward=True)

    # all_info13 = get_analysis_data(csv_list1, csv_list3, gt_string=gtstr,
    #                             invert=True, outward=True)
    # cast_info13 = get_analysis_data(csv_list1, csv_list3, col=3,
    #                             gt_string=gtstr, invert=True, outward=True)
    # turn_info13 = get_analysis_data(csv_list1, csv_list3, col=4,
    #                         gt_string=gtstr, invert=True, outward=True)

    all_info23 = get_analysis_data(csv_list2, csv_list3, gt_string=gtstr,
                                invert=True, outward=True)
    cast_info23 = get_analysis_data(csv_list2, csv_list3, col=3, gt_string=gtstr,
                                invert=True, outward=True)
    turn_info23 = get_analysis_data(csv_list2, csv_list3, col=4,  gt_string=gtstr,
                                invert=True, outward=True)
    accept_info23 = get_analysis_data(csv_list2, csv_list3, col=5,  gt_string=gtstr,
                            invert=True, outward=True)
    dirless_info23 = get_analysis_data(csv_list2, csv_list3, col=6,  gt_string=gtstr,
                            invert=True, outward=True)

# retrieve and plot metrics

## confusion matrices

In [ ]:
def get_confusion_matrix(info):
    cm = []
    for key in info.keys():
        cmatrix = info[key].get('conf_matrix', None)
        if cmatrix is not None:
            cm.append(cmatrix)
    return cm

def condition_cm(dictionary, n = 5):
    cm = get_confusion_matrix(dictionary)
    final_cm = np.zeros((n,n))
    for i in range(len(cm)):
        final_cm += cm[i]
    final_cm = final_cm.astype(int)
    return final_cm

def rm_diags(cm):
    cm_no_diag = cm.copy()
    np.fill_diagonal(cm_no_diag, 0)
    return cm_no_diag

### full confusion matrix

In [ ]:
final_cm = condition_cm(all_info12)
annot_labels = ['straight', 'left cast', 'left turn', 'right cast', 'right turn']
norm_cm = final_cm
final_cm = rm_diags(norm_cm)
norm_cm = final_cm / final_cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots()
sns.heatmap(norm_cm, annot=True, fmt='.2f', xticklabels=annot_labels, yticklabels=annot_labels, cmap='Blues', ax=ax)
ax.set_xlabel('Tanish')
ax.xaxis.set_label_position('top')
ax.xaxis.tick_top()
plt.ylabel('Nitesh')
plt.show(block=False)

### cast CM

In [ ]:
cm = condition_cm(cast_info12)
cast_cm = cm[:4, :4]
cast_cm = np.delete(cast_cm, 2, axis=1)
cast_cm = np.delete(cast_cm, 2, axis=0)

In [ ]:
annot_labels = ['straight', 'left cast', 'right cast']
norm_cm = cast_cm
norm_cm = rm_diags(norm_cm)
# norm_cm = cast_cm / cast_cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots()
sns.heatmap(norm_cm, annot=True, fmt='.2f', xticklabels=annot_labels, yticklabels=annot_labels, cmap='Blues', ax=ax)
ax.set_xlabel('Tanish')
ax.xaxis.set_label_position('top')
ax.xaxis.tick_top()
plt.ylabel('Nitesh')
plt.show(block=False)

### turn CM

In [ ]:
cm = condition_cm(turn_info12)
turn_cm = np.delete(cm, [1,3], axis=1)
turn_cm = np.delete(turn_cm, [1,3], axis=0)

In [ ]:
annot_labels = ['straight', 'left turn', 'right turn']
norm_cm = turn_cm
norm_cm = rm_diags(norm_cm)
# norm_cm = norm_cm / norm_cm.sum(axis=0, keepdims=True)
fig, ax = plt.subplots()
sns.heatmap(norm_cm, annot=True, fmt='.2f', xticklabels=annot_labels, yticklabels=annot_labels, cmap='Blues', ax=ax)
ax.set_xlabel('Tanish')
ax.xaxis.set_label_position('top')
ax.xaxis.tick_top()
plt.ylabel('Nitesh')
plt.show(block=False)

In [ ]:
acc_cm = condition_cm(accept_info12, n=3)

annot_labels = ['N/A', 'accept', 'reject']
norm_cm = acc_cm
# norm_cm = rm_diags(norm_cm)
norm_cm = norm_cm / norm_cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots()
sns.heatmap(norm_cm, annot=True, fmt='.2f', xticklabels=annot_labels, yticklabels=annot_labels, cmap='Blues', ax=ax)
ax.set_xlabel('Tanish')
ax.xaxis.set_label_position('top')
ax.xaxis.tick_top()
plt.ylabel('Nitesh')
plt.show(block=False)

In [ ]:
dir_cm = condition_cm(dirless_info12, n = 4)

annot_labels = ['straight', 'cast', 'turn', 'cast + turn']
norm_cm = dir_cm
norm_cm = rm_diags(norm_cm)
norm_cm = norm_cm / norm_cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots()
sns.heatmap(norm_cm, annot=True, fmt='.2f', xticklabels=annot_labels, 
            yticklabels=annot_labels, cmap='Blues', ax=ax)
ax.set_xlabel('Tanish')
ax.xaxis.set_label_position('top')
ax.xaxis.tick_top()
plt.ylabel('Nitesh')
plt.show(block=False)

### accuracy

In [ ]:
def get_match_score(info):
    best_match = []
    alignment_score = []
    for key in info.keys():
        best_match.append(info[key]['comparison']['best_match'])
        alignment_score.append(info[key]['comparison']['alignment_score'])
    return np.array(best_match), np.array(alignment_score)

In [ ]:
print_dict_tree(dirless_info12[1])


In [ ]:
accuracy_12 = []
accuracy_23 = []
for key in turn_info12.keys():
    accuracy_12.append(turn_info12[key]['accuracy'])
    accuracy_23.append(turn_info23[key]['accuracy'])


plt.boxplot([accuracy_12, accuracy_23], tick_labels=[label1, label3], showfliers=False)
plt.ylabel('Match Accuracy')
plt.xlabel('Quantile Threshold')
plt.title(f'Accuracy of turn label match with peristalsis - {cond}')
plt.ylim(0.3, 1)
plt.show()

In [ ]:
accuracy_12 = []
accuracy_23 = []
for key in cast_info12.keys():
    accuracy_12.append(cast_info12[key]['accuracy'])
    accuracy_23.append(cast_info23[key]['accuracy'])


plt.boxplot([accuracy_12, accuracy_23], tick_labels=[label1, label3], showfliers=False)
plt.ylabel('Match Accuracy')
plt.xlabel('Quantile Threshold')
plt.title(f'Accuracy of cast label match with peristalsis - {cond}')
plt.ylim(0.3, 1)
plt.show()

In [ ]:
accuracy_12 = []
accuracy_23 = []
for key in all_info12.keys():
    accuracy_12.append(all_info12[key]['accuracy'])
    accuracy_23.append(all_info23[key]['accuracy'])

In [ ]:

plt.boxplot([accuracy_12, accuracy_23], tick_labels=[label1, label3], showfliers=False)
plt.ylabel('Match Accuracy')
plt.xlabel('Quantile Threshold')
plt.title(f'Accuracy of label match with peristalsis - {cond}')
plt.ylim(0.3, 1)
plt.show()

In [ ]:
bm12, as12 = get_match_score(turn_info12)
bm13, as13 = get_match_score(turn_info23)

In [ ]:
plt.boxplot([bm12, bm13], tick_labels=[label1, label3], showfliers=False)
plt.ylim(0.2, 1)
plt.ylabel('Best Match Score')
plt.xlabel('Quantile Threshold')
plt.title(f'Best Match Score with peristalsis labels - {cond}')
plt.show()

In [ ]:
plt.boxplot([as12, as13], tick_labels=[label1, label3], showfliers=False)
plt.ylabel('Alignment Score')
plt.xlabel('Quantile Threshold')
plt.title(f'Alignment Score with peristalsis turn labels - {cond}')
plt.ylim(0, 120)
plt.show()

## number of segment figures

In [ ]:
seg_lengths_3 = {0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: []}
seg_lengths_2 = {0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: []}
seg_lengths_1 = {0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: []}

counts_3 = {0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: []}
counts_2 = {0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: []}
counts_1 = {0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: []}

for k in all_info12.keys():
    for key in all_info12[k].keys():
    
        if key == 'Nitesh':
            seg_keys = all_info12[k][key]['seg_lengths'].keys()
            second_key = [k for k in seg_keys if 'invert' in k][0]

            for n in all_info12[k][key]['seg_lengths'][second_key].keys():
                seg_lengths_1[n].extend(all_info12[k][key]['seg_lengths'][second_key][n])
                counts_1[n].append(all_info12[k][key]['counts'][n])

        elif key == 'Tanish':
            seg_keys = all_info12[k][key]['seg_lengths'].keys()
            second_key = [k for k in seg_keys if 'Tanish' in k][0]

            for n in all_info12[k][key]['seg_lengths'][second_key].keys():
                seg_lengths_2[n].extend(all_info12[k][key]['seg_lengths'][second_key][n])
                counts_2[n].append(all_info12[k][key]['counts'][n])

            for n in all_info23[k][key]['seg_lengths'][second_key].keys():
                seg_lengths_3[n].extend(all_info23[k][key]['seg_lengths'][second_key][n])
                counts_3[n].append(all_info23[k][key]['counts'][n])

In [ ]:

plt.figure(figsize=(10, 6))
box1 = plt.boxplot([counts_3[key] for key in counts_3.keys()], 
                   positions=[0.2, 1.2, 2.2, 3.2, 4.2, 5.2, 6.2], widths=0.15, patch_artist=True, 
                   showfliers=False, label=label3)
for patch in box1['boxes']:
    patch.set_facecolor('#ADD8E6')  # Set color for the first boxplot

# Create the third boxplot and set its color
box3 = plt.boxplot([counts_2[key] for key in counts_2.keys()], 
                   positions=[0.4, 1.4, 2.4, 3.4, 4.4, 5.4, 6.4], widths=0.15, 
                   patch_artist=True, showfliers=False, label=label2)
for patch in box3['boxes']:
    patch.set_facecolor('#FFB6C1')  # Set color for the third boxplot

box4 = plt.boxplot([counts_1[key] for key in counts_1.keys()],
                   positions=[0.6, 1.6, 2.6, 3.6, 4.6, 5.6, 6.6], widths=0.15,
                   patch_artist=True, showfliers=False, label = label1)
for patch in box4['boxes']:
    patch.set_facecolor('#FFD700')

plt.xticks([0.3, 2.3, 3.3, 5.3, 6.3], 
           labels = ['straight', 'left cast', 'left turn', 'right cast', 'right turn'])
plt.legend()
plt.title(f'Number of segments - {cond}')
plt.ylabel('Number of segments per trajectory')
plt.show()


In [ ]:
#cast info

counts_3 = {0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: []}
counts_2 = {0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: []}
counts_1 = {0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: []}

for k in cast_info12.keys():
    for key in cast_info12[k].keys():
    
        if key == 'Nitesh':
            seg_keys = cast_info12[k][key]['seg_lengths'].keys()
            second_key = [k for k in seg_keys if 'invert' in k][0]

            for n in cast_info12[k][key]['seg_lengths'][second_key].keys():
                counts_1[n].append(cast_info12[k][key]['counts'][n])

        elif key == 'Tanish':
            seg_keys = cast_info12[k][key]['seg_lengths'].keys()
            second_key = [k for k in seg_keys if 'Tanish' in k][0]

            for n in cast_info12[k][key]['seg_lengths'][second_key].keys():
                counts_2[n].append(cast_info12[k][key]['counts'][n])

            for n in cast_info23[k][key]['seg_lengths'][second_key].keys():
                counts_3[n].append(cast_info23[k][key]['counts'][n])

plt.figure(figsize=(8, 6))
box1 = plt.boxplot([counts_3[key] for key in counts_3.keys()], 
                   positions=[0.2, 1.2, 2.2, 3.2, 4.2, 5.2, 6.2], widths=0.15, patch_artist=True, 
                   showfliers=False, label=label3)
for patch in box1['boxes']:
    patch.set_facecolor('#ADD8E6')  # Set color for the first boxplot

# Create the third boxplot and set its color
box3 = plt.boxplot([counts_2[key] for key in counts_2.keys()], 
                   positions=[0.4, 1.4, 2.4, 3.4, 4.4, 5.4, 6.4], widths=0.15, 
                   patch_artist=True, showfliers=False, label=label2)
for patch in box3['boxes']:
    patch.set_facecolor('#FFB6C1')  # Set color for the third boxplot

box4 = plt.boxplot([counts_1[key] for key in counts_1.keys()],
                   positions=[0.6, 1.6, 2.6, 3.6, 4.6, 5.6, 6.6], widths=0.15,
                   patch_artist=True, showfliers=False, label = label1)
for patch in box4['boxes']:
    patch.set_facecolor('#FFD700')

plt.xticks([0.3, 2.3, 3.3, 5.3, 6.3], 
           labels = ['straight', 'left cast', '', 'right cast', ''])
plt.legend()
plt.title(f'Number of cast segments - {cond}')
plt.ylabel('Number of segments per trajectory')
plt.show()

In [ ]:
#turn info

counts_3 = {0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: []}
counts_2 = {0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: []}
counts_1 = {0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: []}

for k in turn_info12.keys():
    for key in turn_info12[k].keys():
    
        if key == 'Nitesh':
            seg_keys = turn_info12[k][key]['seg_lengths'].keys()
            second_key = [k for k in seg_keys if 'invert' in k][0]

            for n in turn_info12[k][key]['seg_lengths'][second_key].keys():
                counts_1[n].append(turn_info12[k][key]['counts'][n])

        elif key == 'Tanish':
            seg_keys = turn_info12[k][key]['seg_lengths'].keys()
            second_key = [k for k in seg_keys if 'Tanish' in k][0]

            for n in turn_info12[k][key]['seg_lengths'][second_key].keys():
                counts_2[n].append(turn_info12[k][key]['counts'][n])

            for n in turn_info23[k][key]['seg_lengths'][second_key].keys():
                counts_3[n].append(turn_info23[k][key]['counts'][n])

plt.figure(figsize=(8, 6))
box1 = plt.boxplot([counts_3[key] for key in counts_3.keys()], 
                   positions=[0.2, 1.2, 2.2, 3.2, 4.2, 5.2, 6.2], widths=0.15, patch_artist=True, 
                   showfliers=False, label=label3)
for patch in box1['boxes']:
    patch.set_facecolor('#ADD8E6')  # Set color for the first boxplot

box4 = plt.boxplot([counts_1[key] for key in counts_1.keys()],
                   positions=[0.4, 1.4, 2.4, 3.4, 4.4, 5.4, 6.4], widths=0.15,
                   patch_artist=True, showfliers=False, label = label1)
for patch in box4['boxes']:
    patch.set_facecolor('#FFB6C1')  # Set color for the third boxplot

# Create the third boxplot and set its color
box3 = plt.boxplot([counts_2[key] for key in counts_2.keys()], 
                   positions=[0.6, 1.6, 2.6, 3.6, 4.6, 5.6, 6.6], widths=0.15, 
                   patch_artist=True, showfliers=False, label=label2)
for patch in box3['boxes']:
    patch.set_facecolor('#FFD700')


plt.xticks([0.3, 2.3, 3.3, 5.3, 6.3], 
           labels = ['straight', '', 'left turn', '', 'right turn'])
plt.legend()
plt.title(f'Number of turn segments - {cond}')
plt.ylabel('Number of segments per trajectory')
plt.show()

## not useful when looking at peristaltic cycle labels

In [ ]:
# label = 0
# # behavior = 'Straight'

# plt.figure(figsize=(10, 6))
# plt.title(f' - {condition} lengths')
# plt.hist(seg_lengths_peristalsis[label], bins=30, alpha = 0.9, label='peristalsis', density=True)
# plt.hist(seg_lengths_50[label], bins=30, alpha = 0.5, label='supervised 50 quantile', density=True)
# # plt.hist(seg_lengths_625[label], bins=35, alpha = 0.5, label='supervised 62.5 quantile', density=True)
# # plt.hist(seg_lengths_75[label], bins=35, alpha = 0.5, label='supervised 75 quantile', density=True)
# plt.yscale('log')
# # plt.xlim(-1, 800)
# plt.legend()
# plt.show()

In [ ]:
# plt.figure(figsize=(10, 6))

# box1 = plt.boxplot([seg_lengths_peristalsis[label] for label in seg_lengths_peristalsis.keys()], 
#             positions=[0.6, 1.6, 2.6, 3.6, 4.6, 5.6, 6.6], widths=0.15,
#                    patch_artist=True, showfliers=False, label = 'Peristalsis')
# for patch in box1['boxes']:
#     patch.set_facecolor('#FFD700')

# box2 = plt.boxplot([seg_lengths_75[key] for key in seg_lengths_75.keys()], 
#                    positions=[0, 1, 2, 3, 4, 5, 6], widths=0.15, patch_artist=True, 
#                    showfliers=False, label='75 Quantile')
# for patch in box2['boxes']:
#     patch.set_facecolor('#ADD8E6')  # Set color for the first boxplot

# # # Create the second boxplot and set its color
# # box3 = plt.boxplot([seg_lengths_625[key] for key in seg_lengths_625.keys()], 
# #                    positions=[0.2, 1.2, 2.2, 3.2, 4.2, 5.2, 6.2], widths=0.15, 
# #                    patch_artist=True, showfliers=False, label='62.5 Quantile')
# # for patch in box3['boxes']:
# #     patch.set_facecolor('#90EE90')  # Set color for the second boxplot

# # Create the third boxplot and set its color
# box4 = plt.boxplot([seg_lengths_50[key] for key in seg_lengths_50.keys()], 
#                    positions=[0.4, 1.4, 2.4, 3.4, 4.4, 5.4, 6.4], widths=0.15, 
#                    patch_artist=True, showfliers=False, label='50 Quantile')
# for patch in box4['boxes']:
#     patch.set_facecolor('#FFB6C1')  # Set color for the third boxplot



# plt.xticks([0.3, 2.3, 3.3, 5.3, 6.3], 
#            labels = ['straight', 'left cast', 'left turn', 'right cast', 'right turn'])
# plt.legend()
# plt.title(f'Length of label segments - {condition}')
# plt.ylabel('Length of behavioral segment')
# plt.show()

# # plt.figure(figsize=(10, 6))
# # plt.title(f'Left Increase - {behavior} lengths')
# # plt.hist(seg_lengths_peristalsis[label], bins=50, alpha = 0.9, label='peristalsis', density=True)
# # plt.hist(seg_lengths_50[label], bins=40, alpha = 0.5, label='supervised 50 quantile', density=True)
# # plt.hist(seg_lengths_625[label], bins=40, alpha = 0.5, label='supervised 62.5 quantile', density=True)
# # plt.hist(seg_lengths_75[label], bins=40, alpha = 0.5, label='supervised 75 quantile', density=True)

In [ ]:
# for k in seg_lengths_peristalsis.keys():
#     print(k, np.nanmean(seg_lengths_peristalsis[k]), np.nanstd(seg_lengths_peristalsis[k]), np.nanmedian(seg_lengths_peristalsis[k]))

In [ ]:
# for k in seg_lengths_625.keys():
#     print(k, np.nanmean(seg_lengths_625[k]), np.nanstd(seg_lengths_625[k]), np.nanmedian(seg_lengths_625[k]))

# Misc

In [ ]:
# def merge_columns_in_csv(file_path):
#     """
#     Reads a CSV file with 3 columns, processes columns 1 and 2, and writes the updated data back to the same file.
#     If column 2 has a zero and column 1 has a nonzero integer, the nonzero value is moved to column 2, and column 1 is set to 0.
    
#     Args:
#         file_path (str): Path to the CSV file.
#     """
#     # Read the CSV file
#     df = pd.read_csv(file_path, header=None)
    
#     # Ensure columns 1 and 2 are integers
#     df[1] = df[1].astype(int)
#     df[2] = df[2].astype(int)
    
#     # Process the columns
#     mask = (df[2] == 0) & (df[1] != 0)
#     df.loc[mask, 2] = df.loc[mask, 1]  # Move nonzero values from column 1 to column 2
#     df.loc[mask, 0] = 0  # Set column 1 to 0 for those rows
    
#     # Write the updated data back to the same file
#     df.to_csv(file_path, header=False, index=False)

In [ ]:
# for i in range(len(taillocked)):
#     merge_columns_in_csv(taillocked[i])
#     print(taillocked[i])
#     print(np.unique(get_all_labels(taillocked[i], col=2)))
#     print('------------------')